# boolean-mask-identity-replace — ex10: stabilize a normalizing-flow Jacobian batch with identity replace

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-identity-replace`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Boolean mask + identity replace — quick refresher

A boolean mask `m` of the same shape (or broadcastable to) `x` selects elements where `m == True`. Indexed assignment `x[m] = value` replaces those positions. The 'identity replace' pattern: for a batch of matrices, replace the singular / NaN ones with the identity matrix so downstream `solve` / `inverse` calls succeed without short-circuiting.

### Exercise 10 — stabilize a normalizing-flow Jacobian batch with identity replace

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Bloom level: Create
> LO: Compose three boolean masks (NaN, near-singular, sign-flip) into a single 'bad-Jacobian' indicator and substitute the identity matrix into the bad slots so a downstream log-determinant computation stays finite.
> Keywords: normalizing-flow, jacobian, log-det, numerical-stability, integrative, ml-adjacent
> ```

**KCs targeted:** `mask-from-condition`, `mask-update-in-a-loop`

Existing ex1–ex9 covered mask-from-comparison, masked assignment, identity substitution, NMS, attention masks, padding-mean, outlier removal. This drill is the *normalizing-flow* facet: a single end-to-end pipeline that combines THREE bad-condition masks (NaN, near-singular det, sign-flipped det) and identity-replaces the bad Jacobians so `logdet` stays finite.

Implement `ex10_stabilize_jacobian_logdet(jacobians, eps)`:

1. `jacobians` has shape `(B, D, D)` — one Jacobian per batch element. Some may contain NaN, may be near-singular (`|det| < eps`), or may have flipped sign (`det < 0` indicates an orientation reversal we don't trust for this flow).
2. Build three masks of shape `(B,)`:
   - `nan_mask`: any NaN in the matrix
   - `singular_mask`: `|det| < eps`
   - `flip_mask`: `det < 0`
3. Combine via `bad = nan_mask | singular_mask | flip_mask`.
4. Replace the bad Jacobians with the `(D, D)` identity matrix (det = 1, logdet = 0).
5. Return `(stabilized, logdet, bad)` — the cleaned `(B, D, D)` tensor, the `(B,)` `logdet`, and the `(B,)` bool mask of slots that were replaced.

Output dtypes: `stabilized` float32, `logdet` float32, `bad` bool. The bad slots must have `logdet == 0`.

In [ ]:
def ex10_stabilize_jacobian_logdet(jacobians: Tensor, eps: float = 1e-6):
    """Return (stabilized, logdet, bad). Replaces bad Jacobians with I."""
    raise NotImplementedError()


def _test_ex10():
    # Construct a controlled batch.
    good = t.tensor([[2.0, 0.0], [0.0, 3.0]])              # det=6
    near_singular = t.tensor([[1e-9, 0.0], [0.0, 1.0]])    # det≈0
    sign_flip = t.tensor([[0.0, 1.0], [1.0, 0.0]])         # det=-1
    nan_mat = t.tensor([[float('nan'), 0.0], [0.0, 1.0]])
    rot = t.tensor([[0.6, -0.8], [0.8,  0.6]])              # det=1
    jacs = t.stack([good, near_singular, sign_flip, nan_mat, rot]).to(t.float32)

    stabilized, logdet, bad = ex10_stabilize_jacobian_logdet(jacs, eps=1e-4)
    assert stabilized.shape == jacs.shape
    assert stabilized.dtype == t.float32
    assert logdet.shape == (5,)
    assert bad.dtype == t.bool
    assert bad.shape == (5,)
    # Slots 1, 2, 3 should be flagged bad; slots 0 and 4 are clean.
    assert bad.tolist() == [False, True, True, True, False], f'bad mask wrong: {bad.tolist()}'
    # Bad slots got identity (det=1, logdet=0).
    assert t.allclose(stabilized[1], t.eye(2))
    assert t.allclose(stabilized[2], t.eye(2))
    assert t.allclose(stabilized[3], t.eye(2))
    assert abs(logdet[1].item()) < 1e-5
    assert abs(logdet[2].item()) < 1e-5
    assert abs(logdet[3].item()) < 1e-5
    # Good slots are untouched.
    assert t.allclose(stabilized[0], good)
    assert t.allclose(stabilized[4], rot)
    # Good logdets are correct.
    import math
    assert abs(logdet[0].item() - math.log(6.0)) < 1e-4
    assert abs(logdet[4].item() - 0.0) < 1e-3   # det(rot)=1, logdet=0
    # No NaN anywhere in logdet.
    assert t.isfinite(logdet).all(), f'logdet contains non-finite: {logdet}'

    # Scale up: 200 random Jacobians, inject 10% bad.
    rng = t.Generator().manual_seed(0)
    B = 200
    big = t.eye(3).unsqueeze(0).expand(B, 3, 3).clone()
    noise = 0.3 * t.randn(B, 3, 3, generator=rng)
    big = big + noise
    bad_idx = t.randperm(B, generator=rng)[:20]
    big[bad_idx[:7]] = float('nan')             # NaN injection
    big[bad_idx[7:14]] = 1e-12                   # near-singular
    big[bad_idx[14:]] = -big[bad_idx[14:]]       # flip sign by negation (det flips sign for odd D)

    stab, ld, bd = ex10_stabilize_jacobian_logdet(big, eps=1e-4)
    assert t.isfinite(ld).all(), 'no NaN allowed in logdet'
    # Every bad slot has logdet 0.
    assert t.allclose(ld[bd], t.zeros(bd.sum().item()), atol=1e-5)
    # At least the 20 injected bad slots are flagged (we may also catch some natural-near-singular ones).
    for i in bad_idx.tolist():
        assert bd[i].item(), f'injected-bad slot {i} not flagged'

    # --- Visualization: bar chart of logdet, flagged bad slots highlighted ---
    fig, ax = plt.subplots(figsize=(8, 3))
    colors = ['tab:red' if b else 'tab:blue' for b in bd.tolist()]
    ax.bar(range(B), ld.numpy(), color=colors, edgecolor='none')
    ax.set_xlabel('batch index'); ax.set_ylabel('logdet (red = replaced with I)')
    ax.set_title(f'ex10 stabilised logdet — {bd.sum().item()}/{B} replaced')
    ax.axhline(0, color='k', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex10')
    print("ex10 ✓")

_test_ex10()

<details><summary>Solution</summary>

```python
def ex10_stabilize_jacobian_logdet(jacobians: Tensor, eps: float = 1e-6):
    B, D, _ = jacobians.shape
    # Three diagnostic masks of shape (B,).
    nan_mask = t.isnan(jacobians).any(dim=(-2, -1))
    # det of NaN matrices is NaN; mask them off first to compute det safely.
    safe_for_det = jacobians.clone()
    safe_for_det[nan_mask] = t.eye(D)
    dets = t.linalg.det(safe_for_det)
    singular_mask = dets.abs() < eps
    flip_mask     = dets < 0
    bad = nan_mask | singular_mask | flip_mask
    # Identity-replace.
    stabilized = jacobians.clone()
    stabilized[bad] = t.eye(D)
    # logdet of the cleaned batch — guaranteed finite for det > 0.
    logdet = t.log(t.linalg.det(stabilized).clamp(min=eps))
    return stabilized.to(t.float32), logdet.to(t.float32), bad
```

**Compose multiple masks with `|`.** Each diagnostic (NaN, near-singular, flipped) is a single bool tensor; stacking them with bitwise-or gives one combined predicate that drives a single indexed assignment `stabilized[bad] = I`. This pattern is everywhere in production ML — guarding linalg ops, masking dead neurons, gating MoE routes.

**Why we clone before det.** `t.linalg.det` of a matrix with NaN entries returns NaN, which propagates through `<` comparisons in surprising ways. Substituting `I` for NaN slots BEFORE det makes the singular/flip masks clean — even though those slots get re-substituted with `I` again afterwards.

**Why log-det not det.** Normalizing flows accumulate log-Jacobian-determinants across many layers. Working in log-space avoids the under/overflow that comes from multiplying many `det`s; identity-replacement guarantees `log(det) = 0` for the slots we couldn't trust, so the training loss stays finite.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()